# Per-space display layout with unchanged calibrated inference


Scope

Fix the Pf display recipe, which used 20 features through a legacy fallback. Keep the inference grouping used by the shipped calibration in a separately named compatibility recipe. This changes visualization, not calibrated strategy inputs. Reference fingerprints were captured from nightly 0a5f48d before changing the recipe; the earlier dependency experiment records why changing the shared default was insufficient.

In [1]:
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
from starplast import organisms as O, strategies as S, embedding as E
from scripts.rebuild_layouts import rebuild
reference = {'Tg': {'blocks': {'None': '374af1fa255c94ec18472deb83ca39ca0f6ac4bed657a275d1ff7d2a3a201264', 'compartment': '581082c22fc6585dc37395481e82aea2adb5506ac9b5889df0bfcb9057d7e305', 'lopit_unified': '581082c22fc6585dc37395481e82aea2adb5506ac9b5889df0bfcb9057d7e305', 'dtm_class': '98c48a9961a3c387efb923735f728b0afe87c824723406c8e1835320a5de60cb', 'screen_any_phenotype': '374af1fa255c94ec18472deb83ca39ca0f6ac4bed657a275d1ff7d2a3a201264', 'stage_enriched_derived': 'a05bd013f911df6da7fe8abe136a13de528990d62ad7fd3fc30616d2d6f88834', 'fit_invitro_hff': 'c1f1fc352f590b6586824f84adf5eaa7299e7f637d4fbf1758ee4ab957e11107', 'fit_invivo_PE': 'b5de9081ddb5706b72710ae89d0fb9502bd1ab042f1722b7ccbdf6e151bd9d1b', 'expr_tachy': '26c78a5c5adea3ef1642b6a8222ccecaab5a563db1f8bb619bb6b6a6eb775d54'}, 'display': {'matrix_sha256': '4233d10c80f5ff885dd26698cb1571bfef0982a51be9981aaf5bddef4e8f9b80', 'features': ['expr_tachy', 'Tg_transcription_bradyzoite_tissue_cyst__average', 'Tg_transcription_oocyst_sporozoite__average', 'rna51780_mero_r3', 'rna51780_mero_r4', 'rna51780_mero_r5', 'rna51780_mero_r6', 'sexual_stage_8dpi_log2fc', 'cellcycle19092_async_r1', 'cellcycle19092_async_r2', 'cellcycle19092_blocked_r1', 'cellcycle19092_blocked_r2', 'cellcycle19092_1h_r1', 'cellcycle19092_1h_r2', 'cellcycle19092_2h_r1', 'cellcycle19092_2h_r2', 'cellcycle19092_3h_r1', 'cellcycle19092_3h_r2', 'cellcycle19092_4h_r1', 'cellcycle19092_4h_r2', 'cellcycle19092_5h_r1', 'cellcycle19092_5h_r2', 'cellcycle19092_6h_r1', 'cellcycle19092_6h_r2', 'cellcycle19092_7h_r1', 'cellcycle19092_7h_r2', 'cellcycle19092_8h_r1', 'cellcycle19092_8h_r2', 'cellcycle19092_9h_r1', 'cellcycle19092_9h_r2', 'cellcycle19092_10h_r1', 'cellcycle19092_10h_r2', 'cellcycle19092_11h_r1', 'cellcycle19092_11h_r2', 'cellcycle19092_12h_r1', 'cellcycle19092_12h_r2', 'macrophage_expression_percentile', 'Tg_transcription_in_vivo_brain_acute__average', 'Tg_transcription_in_vivo_brain_chronic__average', 'Tg_transcription_purified_bradyzoite_in_vivo__average', 'brain168465_1d_base_mean', 'brain168465_1d_lfc', 'brain168465_2d_base_mean', 'brain168465_2d_lfc', 'brain168465_4d_base_mean', 'brain168465_4d_lfc', 'brain168465_7d_base_mean', 'brain168465_7d_lfc', 'brain168465_14d_base_mean', 'brain168465_14d_lfc', 'stress_5-1_unstress_red_24', 'stress_5-2_unstress_red_24', 'stress_5-24_stress_green_48', 'stress_5-27_stress_green_48', 'stress_6-1_unstress_red_24', 'stress_6-4_unstress_red_24', 'stress_6-16_stress_green_48', 'stress_6-18_stress_green_48', 'stress_7-1_unstress_red_24', 'stress_7-18_stress_green_48', 'rna245775_parent_prebrady_r1', 'rna245775_parent_prebrady_r2', 'rna245775_parent_prebrady_r3', 'rna99395_extracellular_r1', 'rna99395_extracellular_r2', 'morc_MORC_UT1', 'morc_MORC_UT2', 'morc_MORC_UT3', 'morc_MORC_IAA1', 'morc_MORC_IAA2', 'morc_MORC_IAA3', 'morc_MORC-KD-BFD1-KO_UT1', 'morc_MORC-KD-BFD1-KO_UT2', 'morc_MORC-KD-BFD1-KO_UT3', 'morc_MORC-KD-BFD1-KO_IAA1', 'morc_MORC-KD-BFD1-KO_IAA2', 'morc_MORC-KD-BFD1-KO_IAA3', 'morc_DD-BFD1-Ty_UT1', 'morc_DD-BFD1-Ty_UT2', 'morc_DD-BFD1-Ty_UT3', 'morc_DD-BFD1-Ty_Shield1', 'morc_DD-BFD1-Ty_Shield2', 'morc_DD-BFD1-Ty_Shield3', 'cpsf4rna168155_ut_1', 'cpsf4rna168155_ut_2', 'cpsf4rna168155_iaa_7h_1', 'cpsf4rna168155_iaa_7h_2', 'cpsf4rna168155_iaa_24h_1', 'cpsf4rna168155_iaa_24h_2', 'cpsf4rna168155_iaa_48h_1', 'cpsf4rna168155_iaa_48h_2', 'restriction200962_7th_gt_2_kk31_kim_s1_l001', 'restriction200962_7th_gt_20_kk27_kim_s14_l001', 'restriction200962_7th_gt_22_kk29_kim_s15_l001', 'restriction200962_7th_gt_4_kk32_kim_s2_l001', 'restriction200962_kk17_p2wo_ph8_2_s5', 'restriction200962_kk18_p2wo_ph8_2_s1', 'restriction200962_kk19_p2w_ph8_2_s22', 'restriction200962_kk20_p2w_ph8_2_s16', 'restriction200962_kk21_p5wo_ph8_2_s13', 'restriction200962_kk22_p5wo_ph8_2_s15', 'restriction200962_kk23_p5w_ph8_2_s12', 'restriction200962_kk24_p5w_ph8_2_s11', 'restriction200962_kk5_p2wo_ph7_4_s6', 'restriction200962_kk6_p2wo_ph7_4_s19', 'restriction200962_kk7_p2w_ph7_4_s14', 'restriction200962_kk8_p2w_ph7_4_s23', 'expr_max', 'rpf99395_intracellular_r1', 'rpf99395_intracellular_r2', 'rpf129869_confluent_r1', 'rpf129869_confluent_r2', 'rpf129869_confluent_r3', 'rpf129869_subconfluent_r1', 'rpf129869_subconfluent_r2', 'rpf129869_subconfluent_r3', 'rpf245775_parent_tachy_r1', 'rpf245775_parent_tachy_r2', 'rpf245775_parent_tachy_r3', 'rpf245775_parent_prebrady_r1', 'rpf245775_parent_prebrady_r2', 'rpf245775_parent_prebrady_r3', 'rpf99395_extracellular_r1', 'rpf99395_extracellular_r2', 'te302107_tachy_r1', 'te302107_tachy_r2', 'te99395_intracellular_r1', 'te99395_intracellular_r2', 'te129869_confluent_r1', 'te129869_confluent_r2', 'te129869_confluent_r3', 'te129869_subconfluent_r1', 'te129869_subconfluent_r2', 'te129869_subconfluent_r3', 'te245775_parent_tachy_r1', 'te245775_parent_tachy_r2', 'te245775_parent_tachy_r3', 'te99395_extracellular_r1', 'te99395_extracellular_r2', 'te245775_parent_prebrady_r1', 'te245775_parent_prebrady_r2', 'te245775_parent_prebrady_r3', 'rpf245775_eif12ko_tachy_r1', 'rpf245775_eif12ko_tachy_r2', 'rpf245775_eif12ko_tachy_r3', 'rpf245775_eif12ko_prebrady_r1', 'rpf245775_eif12ko_prebrady_r2', 'rpf245775_eif12ko_prebrady_r3', 'te245775_eif12ko_tachy_r1', 'te245775_eif12ko_tachy_r2', 'te245775_eif12ko_tachy_r3', 'te245775_eif12ko_prebrady_r1', 'te245775_eif12ko_prebrady_r2', 'te245775_eif12ko_prebrady_r3', 'rna245775_eif12ko_tachy_r1', 'rna245775_eif12ko_tachy_r2', 'rna245775_eif12ko_tachy_r3', 'rna245775_eif12ko_prebrady_r1', 'rna245775_eif12ko_prebrady_r2', 'rna245775_eif12ko_prebrady_r3', 'Tg_protein_abundance_tachyzoite__fill', 'oocyst_itraq_115_113', 'oocyst_itraq_116_113', 'oocyst_itraq_115_114', 'oocyst_itraq_116_114', 'oocyst_itraq_115_113_1', 'oocyst_itraq_116_113_1', 'oocyst_itraq_115_114_1', 'oocyst_itraq_116_114_1', 'melting_temperature_tm', 'n_phosphosites', 'Tg_phosphorylation_quantitative__average', 'cdpk1_thiophospho_peptides', 'n_acetylation_sites', 'myristoylation_confidence', 'n_arginine_methylation_sites', 'n_lactylation_sites', 'n_nitrosylation_sites', 'n_ubiquitination_sites', 'n_o_fucosyl_peptides', 'bzsub_A_expr', 'bzsub_B_expr', 'bzsub_C_expr', 'bzsub_D_expr', 'bzsub_E_expr', 'iron_depletion_protein_log2fc', 'iron_depletion_protein_padj', 'iron_depletion_rna_log2fc', 'surface_apicoplast_log2fc', 'surface_apicoplast_stringent', 'surface_mitochondrion_log2fc', 'surface_mitochondrion_stringent', 'surface_er_log2fc', 'surface_er_stringent', 'palmitome_odya_vs_hydroxylamine_log2', 'palmitome_odya_vs_palmitate_log2', 'fit_invitro_hff', 'fit_naive_bmdm', 'fit_ifng', 'fit_invivo_PE', 'fit_invivo_lung', 'fit_invivo_liver', 'fit_invivo_spleen', 'fit_invivo_heart', 'fit_invivo_brain', 'fit_lipid_rich_p8', 'fit_lipid_rich_p4p5', 'fit_lipid_limited_p8', 'fit_lipid_limited_p4p5', 'fit_serum_differential_p8', 'fit_serum_differential_p4p5', 'fit_complete_medium_2025', 'fit_no_glucose', 'fit_no_glutamine', 'fit_glucose_dependence', 'fit_glucose_dependence_fdr', 'oxidative_stress_screen_score', 'fit_hyperlopit_unassigned_invivo_lib1', 'fit_hyperlopit_unassigned_invivo_lib2', 'fit_invivo_young2019', 'crispr_invivo_platform_lfc', 'crispr_gra17ko_phenotype', 'crispr_gra17_synthlethal_delta', 'crispr_gra17_candidate', 'crispr_gra12s1_l2fc_invitro', 'crispr_gra12s1_l2fc_invivo', 'crispr_gra12s1_disco', 'crispr_gra12s2_l2fc_invitro', 'crispr_gra12s2_l2fc_invivo', 'crispr_gra12s2_disco', 'diff_reporter_log2_mNG_over_bulk', 'drug_compounds_tested', 'drug_sensitivity_shifts', 'cuttag_hdac3_promoter_ut', 'Tg_chromatin_state_histone_marks__fill', 'atac_promoter_ut', 'n_m6a_peaks', 'Tg_5_utr_architecture__fill', 'novel_transcript_models', 'mrna_log2_remaining_4h_actinomycin', 'mrna_remaining_5h_actinomycin', 'lopit_prob_map', 'lopit_prob_mcmc', 'Tg_localization_transferred__fill', 'n_tm', 'Tg_exposure_to_host_cytosol__fill', 'secretome_soluble_over_vesicle_log2', 'Tg_interaction_proximity_labelling__fill', 'n_xlink_partners', 'best_model_agreement', 'n_ipms_partners', 'n_struct_similar', 'n_host_targets', 'hosttx_T2', 'hosttx_signature_pc01', 'hosttx_signature_pc02', 'hosttx_signature_pc03', 'hosttx_signature_pc04', 'hosttx_signature_pc05', 'hosttx_signature_pc06', 'hosttx_signature_pc07', 'hosttx_signature_pc08', 'hosttx_signature_pc09', 'hosttx_signature_pc10', 'hosttx_signature_pc11', 'hosttx_signature_pc12', 'hosttx_signature_pc13', 'hosttx_signature_pc14', 'hosttx_signature_pc15', 'hosttx_signature_pc16', 'hosttx_signature_pc17', 'hosttx_signature_pc18', 'hosttx_signature_pc19', 'hosttx_signature_pc20', 'hosttx_signature_norm', 'hosttx_signature_n_de', 'n_interpro', 'mean_plddt', 'has_pf_ortholog', 'paralog_number', 'length', 'snp_total_all_strains', 'snp_nonsynonymous', 'snp_synonymous', 'snp_noncoding', 'snp_stop_codon', 'has_ec', 'resistance_allele_count', 'resistance_compound_count', 'Tg_target_engagement_thermal_shift__average', 'bfd2_rip_log2_ip_over_input', 'antisense_expression_percentile', 'codon_enc', 'codon_gc3', 'codon_cai_ribosomal', 'n_crosslink_partners', 'n_bcell_epitopes', 'Tg_t_cell_epitope_content__average', 'escrt_recruitment_xgboost', 'escrt_recruitment_maxvit', 'crispr_reporter_strain_p8_log2', 'af3_sequence_coverage', 'af3_mean_plddt', 'af3_plddt_q25', 'af3_confident_sequence_fraction', 'af3_very_confident_sequence_fraction', 'af3_low_confidence_modelled_fraction', 'af3_confident_rg_angstrom', 'af3_confident_contacts_per_residue', 'af3_ptm', 'ees_vs_tachyzoite_log2', 'enteric_measurements', 'cyst_wall_max_spectral', 'cyst_wall_n_baits']}}, 'Pf': {'blocks': {'None': '0a1386b1d962b07e42f21af71502df485e0762ff83d50963ca328aa6fc57a891', 'lopit_pf_location': '7857add6a49ac96ef7f8c77437fdf0f07e683dca3dcb5d9d1c45f9921c21591b', 'pb_transferred_phenotype': '0098e58ed71afc7c2935adfba42cc5af1da5472fe4cb9b0defbd6d043bc8a33e', 'stage_enriched_derived': 'a07ee9ef66883c9b1fa7b07818874027822b82d1b50423fd74fa493023fb69f4', 'is_exported': 'e790fce657b27053ca55218d39519d81e3e564f59c5d8754216de9bd84906e94', 'piggybac_mis': '7eb12e1f3be9fc56f6c38b1547ae758d043c539498b084e4efdcf7b5e985e903', 'expr_schizont': 'a07ee9ef66883c9b1fa7b07818874027822b82d1b50423fd74fa493023fb69f4', 'mean_plddt': 'bfcceb801789a6b7d3536bdc10b831cd8c90a3d1302be25e0a70b3220ecfa989'}, 'display': {'matrix_sha256': 'c88b27b28b341f248e32896262d401afe65a71a4e47ceb7cc1010c241b5f78de', 'features': ['expr_ring', 'expr_early_trophozoite', 'expr_late_trophozoite', 'expr_schizont', 'expr_gametocyte_ii', 'expr_gametocyte_v', 'expr_ookinete', 'expr_asexual_blood', 'expr_oocyst', 'expr_sporozoite', 'expr_max', 'length', 'n_tm', 'paralog_number', 'n_interpro', 'has_domain', 'has_signal_peptide', 'is_tm', 'n_phosphosites', 'mean_plddt']}}}

In [2]:
checks = []
for code, expected in reference.items():
    ctx = S.Context.shipped(code)
    for name, fingerprint in expected['blocks'].items():
        target = None if name == 'None' else name
        actual = hashlib.sha256(json.dumps(ctx.blocks(target), sort_keys=True).encode()).hexdigest()
        assert actual == fingerprint, (code, target)
    recipe = E.inference_spec(ctx.nodes)
    matrix, names, rows = E.build_matrix(ctx.nodes, recipe, log=lambda *a: None)
    assert names == expected['display']['features']
    assert hashlib.sha256(matrix.tobytes()).hexdigest() == expected['display']['matrix_sha256']
    display, features, rows = E.build_matrix(ctx.nodes, E.default_spec(ctx.nodes), log=lambda *a: None)
    assert np.isfinite(display).all() and len(rows) == len(ctx.nodes)
    checks.append({'organism': code, 'unchanged_strategy_groupings': len(expected['blocks']),
                   'old_display_features': len(names), 'new_display_features': len(features),
                   'rows': len(ctx.nodes)})
pd.DataFrame(checks)

,organism,unchanged_strategy_groupings,old_display_features,new_display_features,rows
0,Tg,9,314,314,8140
1,Pf,8,20,118,5720


Rebuild only the Pf display

Keep every existing edge array and the exact gene order. Refuse a fallback algorithm. The build records the feature list, recipe, node-file hash, backend and coordinates hash.

In [3]:
graph_path = Path(O.graph_path(O.FALCIPARUM))
with np.load(graph_path, allow_pickle=False) as old:
    before = {key: old[key] for key in old.files}
unchanged_paths = [Path(O.nodes_path(c)) for c in O.codes()] + [Path(O.graph_path(O.TOXOPLASMA)), Path('starplast/data/strategy_calibration.json'), Path('starplast/data/slots.json')]
hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in unchanged_paths}
rebuild(graph_path.parent, 'pf_')
with np.load(graph_path, allow_pickle=False) as after:
    for key, values in before.items():
        if key not in ('xyz', 'layout_metadata'):
            np.testing.assert_array_equal(after[key], values)
    assert np.isfinite(after['xyz']).all() and after['xyz'].shape == before['xyz'].shape
    assert not np.array_equal(after['xyz'], before['xyz'])
    record = json.loads(str(after['layout_metadata']))
for name, fingerprint in hashes.items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == fingerprint, name
{'features': len(record['features']), 'method': record['executed_method'], 'backend': record['backend'], 'coordinates_sha256': record['coordinates_sha256'], 'preserved_edge_arrays': len(before) - 2}

[build_graph] matrix: 5,720 genes x 118 features (median, robust)
[build_graph] UMAP: cuml 26.08.00 on the GPU; this can produce a different map from the CPU path
pf_graph.npz: 5,720 genes, 118 features, umap via cuml


{'features': 118, 'method': 'umap', 'backend': 'cuml', 'coordinates_sha256': 'af0ea550fae8df51fb065527475895cf9f77f5b31cf7824cf588448707379650', 'preserved_edge_arrays': 26}

Result

Every recorded strategy grouping and compatibility input matrix remained unchanged. Only the Pf coordinates and their execution metadata changed; the node tables, edge arrays, slot catalogue, Tg layout and shipped calibration file were preserved. Moving calibrated inference to the new slot grouping remains a separate task requiring a new calibration.
The preserved-array count includes the gene ID array.